# BioNodulo on Google Colab

This notebook launches a temporary BioNodulo instance inside a Colab runtime. Files created in Colab are ephemeral unless you download or save them elsewhere.

BioNodulo is distributed under the BioNodulo Research License. Publication, commercial use, and hosted services require a separate license.

In [ ]:
%cd /content
!test -d BioNodulo || git clone -q --branch bionodulo-collab https://github.com/Classacre/BioNodulo.git
%cd /content/BioNodulo
!git fetch -q origin bionodulo-collab
!git checkout -q bionodulo-collab
!git pull -q --ff-only origin bionodulo-collab
!python -m pip install -q .

## Start BioNodulo

Run this cell to embed BioNodulo below through Colab's port viewer.

Use the embedded app output below. If Colab shows a browser-security warning with a `https://localhost:8000/` link, reopen this notebook from the current GitHub Colab badge or clear the old launch-cell output; that warning is from Colab's deprecated window port helper, not this iframe launch cell.

In [ ]:
import subprocess
import sys
import time
from urllib.request import urlopen
from pathlib import Path
from google.colab import output

workspace = Path('/content/bionodulo_workspace')
workspace.mkdir(exist_ok=True)

def bionodulo_ready():
    try:
        with urlopen('http://127.0.0.1:8000/api/health', timeout=1) as response:
            return response.status == 200
    except Exception:
        return False

if not bionodulo_ready():
    server = subprocess.Popen([
        sys.executable,
        'main.py',
        '--host', '0.0.0.0',
        '--port', '8000',
        '--project-root', str(workspace),
    ])

for _ in range(60):
    if bionodulo_ready():
        break
    else:
        time.sleep(1)
else:
    raise RuntimeError('BioNodulo did not become ready on port 8000.')

output.serve_kernel_port_as_iframe(8000, height=900)